### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [5]:
%pip install grpcio -q

Note: you may need to restart the kernel to use updated packages.


In [9]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
# from langchain.agents import Tool
from langchain_core.tools import Tool
from IPython.display import display, Markdown

import os

# from dotenv import load_dotenv

# load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

In [ ]:
BASE_URL = "https://models.github.ai/inference"
MODEL_GPT_4o_MINI = "gpt-4o-mini"
MODEL_NAME = MODEL_GPT_4o_MINI
API_KEY = "github_pat_***"

os.environ["SERPER_API_KEY"] = "4240c****"

### Start with our Message class

In [11]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [12]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [13]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [18]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [15]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(base_url=BASE_URL, model=MODEL_NAME, api_key=API_KEY, temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(base_url=BASE_URL, model=MODEL_NAME, api_key=API_KEY, temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(base_url=BASE_URL, model=MODEL_NAME, api_key=API_KEY, temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [16]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")


In [19]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [20]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some key reasons in favor of choosing AutoGen for your AI agent project:

1. **Reduced Coordination Complexity**: AutoGen simplifies the communication between multiple agents through natural language handoffs, eliminating the need for custom inter-agent protocols. This can significantly streamline development efforts and reduce complexity.

2. **Human-in-the-Loop Capabilities**: The framework offers robust features that facilitate the integration of human oversight into AI processes. This ensures that human feedback can be incorporated effectively, enhancing performance and safety.

3. **Support for Multiple LLM Configurations**: AutoGen supports various large language model (LLM) configurations, allowing developers to leverage the strengths of different LLMs depending on the specific needs of their project.

4. **Native Tool Integration**: The framework allows for seamless code generation and execution for tool integration, which can simplify the integration of external tools and APIs into your AI agent workflows.

5. **Autonomous and Collaborative Functions**: AutoGen is designed for creating agents that can operate autonomously or collaboratively with human users, making it versatile for different types of applications.

6. **Accelerated Development**: By providing a flexible and user-friendly framework, AutoGen can help speed up the development and research processes for agentic AI applications, allowing teams to focus on innovation rather than repetitive tasks.

Overall, these advantages make AutoGen a compelling choice for developing sophisticated AI Agents efficiently. 

TERMINATE.

## Cons of AutoGen:
Here are some cons of using AutoGen in an AI agent project:

1. **Poor Documentation**: The documentation can be difficult to read, which may hinder understanding and effectively implementing AutoGen in projects.

2. **Insufficient Examples**: A lack of practical examples can make it challenging for developers to grasp how to best utilize the tool in real-world applications.

3. **Functionality Issues**: Some features or functionalities may not work as intended, potentially leading to complications and frustrations during development.

These factors could impact the efficiency and effectiveness of developing AI agents, making it worth considering alternatives. 

TERMINATE



## Decision:

Based on the research provided by the team, I would recommend using AutoGen for the project. 

**Rationale**: The pros of AutoGen, particularly its ability to simplify coordination, support various LLM configurations, and enable seamless tool integration, significantly outweigh the cons related to documentation and functionality. The benefits of reducing complexity, enhancing human-in-the-loop capabilities, and accelerating development processes present a strong case for its adoption. The challenges with documentation and examples can be mitigated through team collaboration and exploration. Overall, the potential for efficient and innovative development of sophisticated AI agents supports the decision to go with AutoGen.

TERMINATE.

In [21]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [22]:
await host.stop()